# Notebook 03: Power Analysis

### Purpose
- Evaluates the sensitivity of this experiment: what effects could it reliably detect, and what would it take to detect smaller ones?
- Connects statistical power directly to business decisions about experiment runtime and sample size requirements.

### Objectives
- Compute the minimum detectable effect (MDE) for visit rate, conversion rate, and spend at the actual sample size.
- Plot power curves and sample size curves to visualize the tradeoffs between precision, sample size, and runtime.
- Demonstrate how spend's skewed distribution limits power relative to binary outcomes, and quantify the improvement from winsorization.
- Translate results into a concrete design recommendation: the sample size and runtime a business would need to detect a given lift.

In [11]:
import pandas as pd
import numpy as np

In [5]:
hillstrom_df = pd.read_csv("../data/raw/hillstrom.csv")

hillstrom_df

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
63995,10,2) $100 - $200,105.54,1,0,Urban,0,Web,Mens E-Mail,0,0,0.0
63996,5,1) $0 - $100,38.91,0,1,Urban,1,Phone,Mens E-Mail,0,0,0.0
63997,6,1) $0 - $100,29.99,1,0,Urban,1,Phone,Mens E-Mail,0,0,0.0
63998,1,5) $500 - $750,552.94,1,0,Surburban,1,Multichannel,Womens E-Mail,0,0,0.0


## MDE for visit rate, conversion rate, and spend

In [ ]:
comparisons = [
    ("Mens E-Mail", "No E-Mail"),
    ("Womens E-Mail", "No E-Mail"),
    ("Mens E-Mail", "Womens E-Mail")
]

visit_counts_df = hillstrom_df.groupby("segment")["visit"].agg(["sum", "count"])

results = []

for group1, group2 in comparisons:

    p1 = visit_counts_df.loc[group1, "sum"] / visit_counts_df.loc[group1, "count"]
    p2 = visit_counts_df.loc[group2, "sum"] / visit_counts_df.loc[group2, "count"]

    n1 = visit_counts_df.loc[group1, "count"]
    n2 = visit_counts_df.loc[group2, "count"]

    var = (p1 * (1 - p1) / n1) + (p2 * (1 - p2) / n2)

    se = np.sqrt(var)

    z_alpha = 1.96  # for α=0.05
    z_beta = 0.84   # for 80% power
    
    mde = se * (z_alpha + z_beta)

    results.append((group1, group2, mde))

results_df = pd.DataFrame(results, columns=["Group 1", "Group 2", "MDE"])
display(results_df)

,Group 1,Group 2,MDE
0,Mens E-Mail,No E-Mail,0.009480
1,Womens E-Mail,No E-Mail,0.009056
2,Mens E-Mail,Womens E-Mail,0.010102


In [17]:
conversion_counts_df = hillstrom_df.groupby("segment")["conversion"].agg(["sum", "count"])

results = []

for group1, group2 in comparisons:

    p1 = conversion_counts_df.loc[group1, "sum"] / conversion_counts_df.loc[group1, "count"]
    p2 = conversion_counts_df.loc[group2, "sum"] / conversion_counts_df.loc[group2, "count"]

    n1 = conversion_counts_df.loc[group1, "count"]
    n2 = conversion_counts_df.loc[group2, "count"]

    var = (p1 * (1 - p1) / n1) + (p2 * (1 - p2) / n2)

    se = np.sqrt(var)

    z_alpha = 1.96
    z_beta = 0.84

    mde = se * (z_alpha + z_beta)

    results.append((group1, group2, mde))
    
results_df = pd.DataFrame(results, columns=["Group 1", "Group 2", "MDE"])
display(results_df)

,Group 1,Group 2,MDE
0,Mens E-Mail,No E-Mail,0.002578
1,Womens E-Mail,No E-Mail,0.002303
2,Mens E-Mail,Womens E-Mail,0.002786
